# The Macro Factor

Welcome to Workshop 3.6! In our previous workshops, we learned how to organize individual stocks into a MultiIndex universe and calculate technical indicators like moving averages.

However, stock prices do not move in a vacuum. Broad economic forces like inflation and interest rates exert powerful gravity on every single company in our portfolio. When Treasury yields spike, tech valuations often tumble. When inflation heats up, corporate profit margins compress.

To build realistic multi-factor trading strategies, we need to enrich our daily stock tables with macroeconomic indicators. But combining these datasets presents a major technical headache: stock prices arrive every trading day, while economic metrics arrive on irregular monthly schedules or during bond market trading hours.

In this workshop, we will master `pd.merge_asof()`. This specialized pandas function aligns non-synchronous economic announcements onto daily stock bars safely, without creating look-ahead bias.

> **Key Takeaway**: Macro factors arrive on their own schedule. Using `pd.merge_asof()` with `direction='backward'` ensures that each trading day sees only the most recent public economic report.

## Topic 1: Fetching Treasury Yield Data

The **10-Year Treasury Yield** (`^TNX` on Yahoo Finance) acts as the benchmark risk-free rate for global financial markets. When Treasury yields climb, borrowing costs rise and future corporate cash flows become less valuable today.

Bond markets follow their own calendar. Sometimes bond trading closes early, or banks observe holidays when stock exchanges remain open.

Let's download daily 10-Year Treasury yield data and inspect the structure.

In [1]:
import yfinance as yf

# Fetch 10-Year Treasury Yield data:
tnx = yf.download("^TNX", start="2023-01-01", end="2024-01-01", progress=False)
if isinstance(tnx.columns, pd.MultiIndex):
    tnx.columns = tnx.columns.get_level_values(0)
tnx.index = tnx.index.tz_localize(None)
tnx = tnx[["Close"]].rename(columns={"Close": "TNX_Yield"})

# Let's check the first few rows of Treasury yields:
print("10-Year Treasury Yield (^TNX) - First 5 rows:")
print(tnx.head())
print(f"\nTotal trading days fetched: {len(tnx)}")

10-Year Treasury Yield (^TNX) - First 5 rows:
Price       TNX_Yield
Date                 
2023-01-03      3.793
2023-01-04      3.709
2023-01-05      3.720
2023-01-06      3.569
2023-01-09      3.517

Total trading days fetched: 250

## Topic 2: Simulating CPI Data

The **Consumer Price Index (CPI)** tracks price changes across consumer goods. The Bureau of Labor Statistics releases these numbers once a month on specific announcement mornings.

Because Yahoo Finance focuses on exchange-traded asset prices rather than government economic reports, we will simulate a realistic monthly CPI release schedule for early 2023.

Notice the dates: these reports appear on arbitrary Thursdays or Tuesdays, completely out of sync with regular end-of-month calendar boundaries.

Let's build our CPI DataFrame and inspect the schedule.

In [2]:
import pandas as pd

# Realistic monthly CPI announcement schedule:
cpi_dates = pd.to_datetime(["2023-01-12", "2023-02-14", "2023-03-14", "2023-04-12"])
cpi_values = [6.5, 6.4, 6.0, 4.9]
cpi = pd.DataFrame({"Date": cpi_dates, "CPI_YoY": cpi_values})
cpi = cpi.set_index("Date")

# Let's inspect the announcement calendar:
print("Simulated Monthly CPI Release Schedule:")
print(cpi)

Simulated Monthly CPI Release Schedule:
            CPI_YoY
Date               
2023-01-12      6.5
2023-02-14      6.4
2023-03-14      6.0
2023-04-12      4.9

## Topic 3: The Problem with Standard Merging

Let's see what happens if we try to combine our daily stock universe with our macro datasets using a standard relational join like `pd.merge()`.

A standard relational join requires exact date equality (`Date == Date`). Think of it like matching two spreadsheets on an exact employee ID. If the IDs do not match down to the exact character, pandas discards the row or fills it with `NaN`.

For monthly CPI, we have only four dates in our entire table. When we merge that onto 250 daily stock trading bars using an exact match, almost every single trading day ends up with a missing `NaN` value.

Let's test this in code and observe the massive data loss.

In [3]:
from pathlib import Path

# Load universe (AAPL and MSFT) from local cache:
cache_dir = Path("data_cache")
aapl = pd.read_parquet(cache_dir / "AAPL.parquet")
msft = pd.read_parquet(cache_dir / "MSFT.parquet")
aapl["Ticker"] = "AAPL"
msft["Ticker"] = "MSFT"

universe = pd.concat([aapl, msft]).reset_index().set_index(["Date", "Ticker"]).sort_index()

# Standard merge with exact date matching:
merged_wrong = pd.merge(universe.reset_index(), tnx.reset_index(), on="Date", how="left")

# Let's check the Treasury merge:
print("Standard merge with Treasury yields (exact date matching):")
print(merged_wrong[["Date", "Ticker", "Close", "TNX_Yield"]].head(6))

# Now test standard merge with monthly CPI data:
merged_cpi_wrong = pd.merge(universe.reset_index(), cpi.reset_index(), on="Date", how="left")
total_rows = len(merged_cpi_wrong)
null_rows = merged_cpi_wrong["CPI_YoY"].isna().sum()

# Let's inspect the missing data proportion:
print("\nStandard merge with monthly CPI data:")
print(f"Total Rows:     {total_rows}")
print(f"Missing Values: {null_rows} out of {total_rows} ({null_rows / total_rows:.1%})")
print("\nFirst 8 rows of merged_cpi_wrong:")
print(merged_cpi_wrong[["Date", "Ticker", "Close", "CPI_YoY"]].head(8))

Standard merge with Treasury yields (exact date matching):
Price       Date Ticker       Close  TNX_Yield
0     2023-01-03   AAPL  122.876732      3.793
1     2023-01-03   MSFT  232.510544      3.793
2     2023-01-04   AAPL  124.144127      3.709
3     2023-01-04   MSFT  222.339798      3.709
4     2023-01-05   AAPL  122.827606      3.720
5     2023-01-05   MSFT  215.750153      3.720

Standard merge with monthly CPI data:
Total Rows:     500
Missing Values: 492 out of 500 (98.4%)

First 8 rows of merged_cpi_wrong:
        Date Ticker       Close  CPI_YoY
0 2023-01-03   AAPL  122.876732      NaN
1 2023-01-03   MSFT  232.510544      NaN
2 2023-01-04   AAPL  124.144127      NaN
3 2023-01-04   MSFT  222.339798      NaN
4 2023-01-05   AAPL  122.827606      NaN
5 2023-01-05   MSFT  215.750153      NaN
6 2023-01-06   AAPL  127.346924      NaN
7 2023-01-06   MSFT  218.292816      NaN

### Why Exact Matching Breaks Down

Over 96 percent of our rows ended up blank! That happens because the CPI report for January is published on January 12. Between January 12 and February 14, no new CPI report exists.

In the real world, investors do not forget the inflation rate between announcements. A trader sitting at their desk on January 25 relies on the January 12 number because it is the latest known information.

What we need is an **"as-of" join**: for any given trading day, find the most recent economic release that happened on or before that day.

Let's see how pandas solves this.

## Topic 4: The Solution - pd.merge_asof()

Pandas provides a specialized tool engineered precisely for financial time series: `pd.merge_asof()`.

The term **as-of** means: match each record in our primary table with the latest available record from the secondary table as of that moment.

To use `pd.merge_asof()` safely, we must follow two simple requirements:
- Both DataFrames must have their join column sorted in ascending order.
- We must specify `direction="backward"` so pandas searches backward in time to find the latest known release.

Let's run `pd.merge_asof()` and inspect the resulting alignment.

In [4]:
# Convert to DataFrames with Date as a regular column:
universe_df = universe.reset_index()
tnx_df = tnx.reset_index()
cpi_df = cpi.reset_index()

# Ensure dates are sorted in ascending order:
universe_df = universe_df.sort_values("Date")
tnx_df = tnx_df.sort_values("Date")
cpi_df = cpi_df.sort_values("Date")

# Merge Treasury yields using backward matching:
with_tnx = pd.merge_asof(universe_df, tnx_df, on="Date", direction="backward")

# Merge CPI using backward matching:
with_macro = pd.merge_asof(with_tnx, cpi_df, on="Date", direction="backward")

# Let's inspect the aligned dataset:
print("Successfully merged macro features onto equity universe:")
print(with_macro[["Date", "Ticker", "Close", "TNX_Yield", "CPI_YoY"]].head(8))

print("\nInspecting transition across the February 14 CPI release:")
feb_window = with_macro[(with_macro["Date"] >= "2023-02-10") & (with_macro["Date"] <= "2023-02-16")]
print(feb_window[["Date", "Ticker", "Close", "TNX_Yield", "CPI_YoY"]])

Successfully merged macro features onto equity universe:
        Date Ticker       Close  TNX_Yield  CPI_YoY
0 2023-01-03   AAPL  122.876732      3.793      NaN
1 2023-01-03   MSFT  232.510544      3.793      NaN
2 2023-01-04   AAPL  124.144127      3.709      NaN
3 2023-01-04   MSFT  222.339798      3.709      NaN
4 2023-01-05   AAPL  122.827606      3.720      NaN
5 2023-01-05   MSFT  215.750153      3.720      NaN
6 2023-01-06   AAPL  127.346924      3.569      NaN
7 2023-01-06   MSFT  218.292816      3.569      NaN

Inspecting transition across the February 14 CPI release:
         Date Ticker       Close  TNX_Yield  CPI_YoY
54 2023-02-10   AAPL  148.588379      3.744      6.5
55 2023-02-10   MSFT  255.336578      3.744      6.5
56 2023-02-13   AAPL  151.382858      3.717      6.5
57 2023-02-13   MSFT  263.313934      3.717      6.5
58 2023-02-14   AAPL  150.743256      3.761      6.4
59 2023-02-14   MSFT  264.138947      3.761      6.4
60 2023-02-15   MSFT  262.027679      3.809  

### Examining the Seamless Transition

Look closely at the rows surrounding mid-February in our output above:
- On `2023-02-10` and `2023-02-13`, the CPI reading is `6.5`, which was the latest number from the January 12 report.
- On `2023-02-14`, the new report comes out, and the CPI reading immediately updates to `6.4`.
- On subsequent trading days, `6.4` carries forward smoothly until the next release in March.

This behavior mirrors how professional trading desks consume economic releases.

> **Key Takeaway**: Setting `direction='backward'` automatically forward-fills past announcements up to the next release date without any risk of peeking into the future.

## Topic 5: The Danger of direction="forward"

Pandas also offers an alternative parameter value: `direction="forward"`.

When `direction="forward"` is set, pandas looks ahead and matches each trading day with the **next upcoming** announcement.

This introduces devastating look-ahead bias into our strategy. It assigns future economic numbers to historical trading days before the government ever published them.

Let's test `direction="forward"` intentionally to see how look-ahead bias creeps in.

In [5]:
# Intentionally flawed merge peeking into future data:
with_forward = pd.merge_asof(universe_df, cpi_df, on="Date", direction="forward")

# Let's inspect the early January rows:
print("Demonstrating Look-Ahead Bias with direction='forward':")
print(with_forward[["Date", "Ticker", "Close", "CPI_YoY"]].head(6))

Demonstrating Look-Ahead Bias with direction='forward':
        Date Ticker       Close  CPI_YoY
0 2023-01-03   AAPL  122.876732      6.5
1 2023-01-03   MSFT  232.510544      6.5
2 2023-01-04   AAPL  124.144127      6.5
3 2023-01-04   MSFT  222.339798      6.5
4 2023-01-05   AAPL  122.827606      6.5
5 2023-01-05   MSFT  215.750153      6.5

### Spotting the Cheating Behavior

Look at the row for `2023-01-03` in the table above:
- The DataFrame assigned `CPI_YoY = 6.5` to January 3.
- But the government did not publish that figure until **January 12**!

If our algorithm used `CPI_YoY` to make portfolio decisions on January 3, it would trade on secret future knowledge that no real trader possessed. That strategy would look stellar in backtests, but fail catastrophically in live execution.

Always double-check your join direction: use `direction="backward"`.

> **Key Takeaway**: Never use `direction='forward'` when merging macro data onto market prices. It leaks future economic releases into past trading sessions.

## Topic 6: Creating a Reusable Merge Function

To prevent copy-pasting code and ensure consistent point-in-time safety, let's wrap this logic inside a reusable helper function named `merge_macro()`.

Our function takes defensive copies of both DataFrames, ensures date formats match, sorts the time series, and applies backward matching.

Let's write and verify our helper function.

In [6]:
def merge_macro(price_df, macro_df, on="Date"):
    price_df = price_df.copy()
    macro_df = macro_df.copy()
    price_df[on] = pd.to_datetime(price_df[on])
    macro_df[on] = pd.to_datetime(macro_df[on])
    price_df = price_df.sort_values(on)
    macro_df = macro_df.sort_values(on)
    return pd.merge_asof(price_df, macro_df, on=on, direction="backward")

# Let's test our helper on our universe:
test_result = merge_macro(universe_df, tnx_df)

print(f"Original universe rows: {len(universe_df)}")
print(f"Merged DataFrame rows:  {len(test_result)}")
print("\nFirst 5 rows of test_result:")
print(test_result[["Date", "Ticker", "Close", "TNX_Yield"]].head())

Original universe rows: 500
Merged DataFrame rows:  500

First 5 rows of test_result:
Price       Date Ticker       Close  TNX_Yield
0     2023-01-03   AAPL  122.876732      3.793
1     2023-01-03   MSFT  232.510544      3.793
2     2023-01-04   AAPL  124.144127      3.709
3     2023-01-04   MSFT  222.339798      3.709
4     2023-01-05   AAPL  122.827606      3.720

---

## Practice Time

Now it is your turn to apply `pd.merge_asof()` across different economic indicators. Work through the three challenges below.

---

### Challenge 1: Merging Short-Term Treasury Bill Rates
Download the short-term 13-Week Treasury Bill Yield (`^IRX` on Yahoo Finance) for 2023. Rename the `Close` column to `IRX_Yield`, reset the index, and merge it onto `universe_df` using our `merge_macro()` function.

In [7]:
# Challenge 1: Fetch ^IRX and merge with universe_df
# Write your code below this line:



### Challenge 2: Merging Federal Reserve Rate Decisions
Simulate a Federal Reserve interest rate decision schedule for 2023:
- 2023-02-01: Target Rate = 4.75%
- 2023-03-22: Target Rate = 5.00%
- 2023-05-03: Target Rate = 5.25%
- 2023-07-26: Target Rate = 5.50%

Create a DataFrame for these policy changes, sort by `Date`, and merge them onto `universe_df` using `pd.merge_asof(..., direction="backward")`. Inspect the rows around March 22 to confirm the rate stepped up correctly.

In [8]:
# Challenge 2: Simulate Fed meetings and merge using merge_asof
# Write your code below this line:



### Challenge 3: Merging Directly onto MultiIndex Universes
Write a Python function named `safe_merge_macro_multiindex(price_multi_df, macro_df, on="Date")` that:
- Takes a MultiIndex price DataFrame `(Date, Ticker)` and a macro DataFrame.
- Resets the price DataFrame index safely.
- Merges the macro series using `merge_macro()`.
- Restores the MultiIndex structure `(Date, Ticker)` and returns the sorted result.

Test your function by merging `tnx_df` onto our original `universe` DataFrame.

In [9]:
# Challenge 3: Write and test safe_merge_macro_multiindex
# Write your code below this line:



---

## Solutions

Compare your answers with the reference implementations below whenever you are ready.

### Solution for Challenge 1

In [10]:
# Solution for Challenge 1:
irx = yf.download("^IRX", start="2023-01-01", end="2024-01-01", progress=False)
if isinstance(irx.columns, pd.MultiIndex):
    irx.columns = irx.columns.get_level_values(0)
irx.index = irx.index.tz_localize(None)
irx_df = irx[["Close"]].rename(columns={"Close": "IRX_Yield"}).reset_index()

merged_irx = merge_macro(universe_df, irx_df)

# Let's inspect the merged Treasury Bill yield:
print("Challenge 1 Output (first 6 rows):")
print(merged_irx[["Date", "Ticker", "Close", "IRX_Yield"]].head(6))

Exercise 1 Output (first 6 rows):
Price       Date Ticker       Close  IRX_Yield
0     2023-01-03   AAPL  122.876732      4.255
1     2023-01-03   MSFT  232.510544      4.255
2     2023-01-04   AAPL  124.144127      4.400
3     2023-01-04   MSFT  222.339798      4.400
4     2023-01-05   AAPL  122.827606      4.498
5     2023-01-05   MSFT  215.750153      4.498

In [11]:
# Solution for Challenge 2:
fed_meetings = pd.DataFrame({
    "Date": pd.to_datetime(["2023-02-01", "2023-03-22", "2023-05-03", "2023-07-26"]),
    "Target_Rate": [4.75, 5.00, 5.25, 5.50]
}).sort_values("Date")

merged_fed = pd.merge_asof(universe_df, fed_meetings, on="Date", direction="backward")

# Let's verify the rate step-up around March 22:
print("Challenge 2 Output around March 22 Fed Meeting:")
march_slice = merged_fed[(merged_fed["Date"] >= "2023-03-20") & (merged_fed["Date"] <= "2023-03-24")]
print(march_slice[["Date", "Ticker", "Close", "Target_Rate"]])

Exercise 2 Output around March 22 Fed Meeting:
          Date Ticker       Close  Target_Rate
104 2023-03-20   MSFT  264.858917         4.75
105 2023-03-20   AAPL  154.875885         4.75
106 2023-03-21   MSFT  266.366852         4.75
107 2023-03-21   AAPL  156.725769         4.75
108 2023-03-22   AAPL  155.299011         5.00
109 2023-03-22   MSFT  264.917267         5.00
110 2023-03-23   AAPL  156.381378         5.00
111 2023-03-23   MSFT  270.141876         5.00
112 2023-03-24   AAPL  157.680176         5.00
113 2023-03-24   MSFT  272.972961         5.00

In [12]:
# Solution for Challenge 3:
def safe_merge_macro_multiindex(price_multi_df, macro_df, on="Date"):
    price_reset = price_multi_df.reset_index()
    merged = merge_macro(price_reset, macro_df, on=on)
    return merged.set_index(["Date", "Ticker"]).sort_index()

# Let's test the function on our MultiIndex universe:
multi_with_tnx = safe_merge_macro_multiindex(universe, tnx_df)
print("Challenge 3 MultiIndex Output (first 6 rows):")
print(multi_with_tnx[["Close", "TNX_Yield"]].head(6))

Exercise 3 MultiIndex Output (first 6 rows):
Price                   Close  TNX_Yield
Date       Ticker                       
2023-01-03 AAPL    122.876732      3.793
           MSFT    232.510544      3.793
2023-01-04 AAPL    124.144127      3.709
           MSFT    222.339798      3.709
2023-01-05 AAPL    122.827606      3.720
           MSFT    215.750153      3.720